In [ ]:

import os
import shutil
import zipfile

destination_dir = "/content/Tea_Leaf_Data_Set"

if os.path.exists(destination_dir):
    if os.path.isdir(destination_dir):
        shutil.rmtree(destination_dir)
    else:
        os.remove(destination_dir)

os.makedirs(destination_dir)

with zipfile.ZipFile("/content/tea_leaf_quality.zip", 'r') as zip_ref:
    zip_ref.extractall(destination_dir)

<h>Image Pre Processing</h>

In [ ]:
import cv2
import os
import numpy as np
from tqdm import tqdm

input_image_dir = '/content/Tea_Leaf_Data_Set/test/images'
input_label_dir = '/content/Tea_Leaf_Data_Set/test/images'
output_image_dir = '/content/Tea_Leaf_Processed_Data_Set/test/images'
output_label_dir = '/content/Tea_Leaf_Processed_Data_Set/test/labels'
resize_to = (640, 640)
include_exg = True
os.makedirs(output_image_dir, exist_ok=True)
os.makedirs(output_label_dir, exist_ok=True)


def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

def apply_bilateral_filter(image):
    return cv2.bilateralFilter(image, d=9, sigmaColor=75, sigmaSpace=75)

def compute_exg(image):
    image = image.astype(np.float32)
    exg = 2 * image[:, :, 1] - image[:, :, 0] - image[:, :, 2]
    exg = np.clip(exg, 0, 255)
    exg = cv2.normalize(exg, None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
    return exg.astype(np.uint8)

image_files = [f for f in os.listdir(input_image_dir) if f.lower().endswith(('.jpg', '.png'))]

print(f'Processing {len(image_files)} images...')

for file_name in tqdm(image_files):
    input_path = os.path.join(input_image_dir, file_name)
    output_path = os.path.join(output_image_dir, file_name)

    image = cv2.imread(input_path)
    if resize_to:
        image = cv2.resize(image, resize_to)

    image = apply_clahe(image)
    image = apply_bilateral_filter(image)

    if include_exg:
        exg = compute_exg(image)
        exg = cv2.merge([exg, exg, exg])
        image = cv2.addWeighted(image, 0.8, exg, 0.2, 0)

    cv2.imwrite(output_path, image)

    label_file = os.path.splitext(file_name)[0] + '.txt'
    label_input_path = os.path.join(input_label_dir, label_file)
    label_output_path = os.path.join(output_label_dir, label_file)

    if os.path.exists(label_input_path):
        with open(label_input_path, 'r') as src, open(label_output_path, 'w') as dst:
            dst.write(src.read())

print('Preprocessing complete.')


Processing 469 images...


100%|██████████| 469/469 [00:37<00:00, 12.37it/s]

✅ Preprocessing complete.


In [ ]:
!zip -r /content/Tea_Leaf_Processed_Data_Set.zip /content/Tea_Leaf_Processed_Data_Set

  adding: content/Tea_Leaf_Processed_Data_Set/ (stored 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/ (stored 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/labels/ (stored 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/images/ (stored 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/images/IMG-20231226-WA0134_jpg.rf.1f2abfa66b5750d670f5617b8fef9e6a.jpg (deflated 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/images/IMG_20231226_150054_jpg.rf.7cea028f79857704f9a7e0f0a31ee1be.jpg (deflated 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/images/20231226_114648_jpg.rf.9119340201b14c9d28dcf21559df5e12.jpg (deflated 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/images/IMG_20231226_142435_jpg.rf.f81ad711aad3c51fe581727767b93839.jpg (deflated 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/images/20231226_115415_jpg.rf.be67e3300fb0484b9c6bd9d09280579a.jpg (deflated 0%)
  adding: content/Tea_Leaf_Processed_Data_Set/test/images/IMG_2

In [ ]:
from google.colab import files
files.download("/content/Tea_Leaf_Processed_Data_Set.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>